# Математическая статистика

# Задание 2 Python
Дан файл со значениями характеристик ВУЗов РФ. Прочитать значения среднего балла ЕГЭ ege_budg и общего дохода ВУЗа total_income, очистить выборку от данных с ege_budg вне диапазона [50, 100].

Построить гистограммы среднего балла ЕГЭ и общего дохода ВУЗа для исходной и очищенной выборок.

Вычислить выборочное среднее и среднеквадратическое отклонение среднего балла ЕГЭ и общего дохода ВУЗа. Вычислить ковариацию и коэффициент корреляции по формуле и с помощью scipy.stats.


Сделать вывод о связи на основании значения ковариации и коэффициента корреляции.

Построить доверительные интервалы с доверительной вероятностью 95% и 98% для матожидания и дисперсии среднего балла ЕГЭ и общего дохода ВУЗа.

# Формулы для анализа

**Выборочное среднее**
$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$, $\bar{y} = \frac{1}{n}\sum_{i=1}^{n} y_i$

**Выборочная дисперсия**
$D_x = \frac{1}{n}\sum_{i=1}^{n} (x_i - \bar{x})^2$, $D_y = \frac{1}{n}\sum_{i=1}^{n} (y_i - \bar{y})^2$

**Ковариация**
$\text{cov}(x,y) = \frac{1}{n}\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})$

**Коэффициент корреляции Пирсона**
$r = \frac{\text{cov}(x,y)}{\sqrt{D_x \cdot D_y}}$

**Критерий значимости**

Если $p\text{-значение} < 0.05$, то связь статистически значима.

# Интерпретация коэффициента корреляции
- $|r| \leq 0.3$ — слабая связь
- $0.3 < |r| \leq 0.7$ — умеренная связь
- $|r| > 0.7$ — сильная связь


Функции pandas
- pd.read_excel() – чтение Excel файла
- df.shape – размерность DataFrame
- df.columns.tolist() – список столбцов
- df['column'].notna() – проверка на не-NaN значения
- df.copy() – копирование DataFrame
- df['col1'].cov(df['col2']) – ковариация
- df['col1'].corr(df['col2']) – коэффициент корреляции

Функции numpy
- np.mean() – среднее значение
- np.sum() – сумма элементов
- np.sqrt() – квадратный корень

Функции scipy.stats
- stats.pearsonr() – коэффициент корреляции Пирсона с p-значением

Функции matplotlib
- plt.subplots() – создание подграфиков
- ax.hist() – построение гистограммы
- ax.set_xlabel(), ax.set_ylabel() – подписи осей
- ax.set_title() – заголовок графика
- ax.axvline() – вертикальная линия
- ax.legend() – легенда графика
- plt.tight_layout() – автоматическая настройка расположения
- plt.savefig() – сохранение графика
- plt.show() – отображение графика

Методы DataFrame и встроенные функции
- all() – проверка всех элементов
- if col not in df.columns – проверка наличия столбца
- .format() и f-строки – форматирование вывода

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

# Проверка текущей директории
print(f'Текущая директория: {os.getcwd()}')
print(f'Файлы в директории: {os.listdir(".")}')


In [ ]:
# ============================================
# БЛОК 1: Извлечение данных из поврежденного data.xlsx
# Используем поиск столбца year (2000-2030), затем смещение
# ============================================
import pandas as pd
import csv
import numpy as np

EXCEL_FILE = 'data.xlsx'
OUTPUT_CSV = 'clean_data.csv'

print("Извлечение данных из data.xlsx...")

# Читаем весь файл без заголовков
df_raw = pd.read_excel(EXCEL_FILE, header=None)
print(f"Размер прочитанных данных: {df_raw.shape}")

# Получаем заголовки из первой строки
header = df_raw.iloc[0].tolist()
print(f"Заголовки: {header[:10]}")

# Находим столбец year по заголовку
try:
    year_col_by_header = header.index('year')
    print(f"Столбец 'year' по заголовку: {year_col_by_header}")
except ValueError:
    year_col_by_header = None
    print("Столбец 'year' не найден в заголовках!")

extracted = []
errors = 0

for row_idx in range(1, len(df_raw)):
    row = df_raw.iloc[row_idx]
    year_col = None
    year_val = None
    
    # Ищем столбец со значением года (2000-2030)
    for col_idx in range(len(row)):
        val = row.iloc[col_idx]
        if val is not None:
            try:
                y = float(val)
                if 2000 <= y <= 2030:
                    year_col = col_idx
                    year_val = val
                    break
            except (ValueError, TypeError):
                pass
    
    if year_col is None:
        errors += 1
        if errors <= 5:
            print(f"Строка {row_idx}: Год не найден!")
        continue
    
    # ege_budg находится на 8 позиций вправо от year
    # total_income находится на 13 позиций вправо от year
    ege_col = year_col + 8
    inc_col = year_col + 13
    
    ege_val = row.iloc[ege_col] if ege_col < len(row) else None
    inc_val = row.iloc[inc_col] if inc_col < len(row) else None
    
    extracted.append({'year': year_val, 'ege_budg': ege_val, 'total_income': inc_val})

print(f"\nИзвлечено строк: {len(extracted)}, ошибок: {errors}")

# Сохраняем в CSV
with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['year', 'ege_budg', 'total_income'])
    writer.writeheader()
    writer.writerows(extracted)

print(f"Данные сохранены в {OUTPUT_CSV}")
df_check = pd.read_csv(OUTPUT_CSV)
print(f"Первые 3 строки:\n{df_check.head(3)}")


In [ ]:
# ============================================
# БЛОК 2: Загрузка подготовленных данных
# ============================================
df = pd.read_csv('clean_data.csv')
df['ege_budg'] = pd.to_numeric(df['ege_budg'], errors='coerce')
df['total_income'] = pd.to_numeric(df['total_income'], errors='coerce')

print("=" * 60)
print("ИСХОДНАЯ ВЫБОРКА")
print("=" * 60)
print(f"Размерность: {df.shape}")
print(f"Столбцы: {df.columns.tolist()}")
print(f"\nПропущенные значения:\n{df.isnull().sum()}")
print(f"\nОписательная статистика:\n{df[['ege_budg', 'total_income']].describe()}")


In [ ]:
# ============================================
# БЛОК 3: Очистка данных (ege_budg вне [50, 100])
# ============================================
print("\n" + "=" * 60)
print("ОЧИСТКА ДАННЫХ")
print("=" * 60)

mask = (df['ege_budg'] >= 50) & (df['ege_budg'] <= 100)
df_clean = df[mask].copy()

print(f"Удалено строк: {len(df) - len(df_clean)}")
print(f"Осталось строк: {len(df_clean)}")
print(f"\nОписательная статистика (очищенная):\n{df_clean[['ege_budg', 'total_income']].describe()}")


In [ ]:
# ============================================
# БЛОК 4: Построение гистограмм
# ============================================
print("\n" + "=" * 60)
print("ПОСТРОЕНИЕ ГИСТОГРАММ")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df['ege_budg'].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['ege_budg'].mean(), color='red', linestyle='--', label=f'Среднее={df["ege_budg"].mean():.2f}')
axes[0, 0].set_xlabel('Средний балл ЕГЭ (ege_budg)')
axes[0, 0].set_ylabel('Частота')
axes[0, 0].set_title('Гистограмма ege_budg (исходная выборка)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(df_clean['ege_budg'].dropna(), bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(df_clean['ege_budg'].mean(), color='red', linestyle='--', label=f'Среднее={df_clean["ege_budg"].mean():.2f}')
axes[0, 1].set_xlabel('Средний балл ЕГЭ (ege_budg)')
axes[0, 1].set_ylabel('Частота')
axes[0, 1].set_title('Гистограмма ege_budg (очищенная выборка)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(df['total_income'].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(df['total_income'].mean(), color='red', linestyle='--', label=f'Среднее={df["total_income"].mean():.0f}')
axes[1, 0].set_xlabel('Общий доход ВУЗа (total_income)')
axes[1, 0].set_ylabel('Частота')
axes[1, 0].set_title('Гистограмма total_income (исходная выборка)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(df_clean['total_income'].dropna(), bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(df_clean['total_income'].mean(), color='red', linestyle='--', label=f'Среднее={df_clean["total_income"].mean():.0f}')
axes[1, 1].set_xlabel('Общий доход ВУЗа (total_income)')
axes[1, 1].set_ylabel('Частота')
axes[1, 1].set_title('Гистограмма total_income (очищенная выборка)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('histograms.png', dpi=150)
print("Гистограммы сохранены в histograms.png")
plt.show()


In [ ]:
# ============================================
# БЛОК 5: Выборочное среднее и среднеквадратическое отклонение
# ============================================
print("\n" + "=" * 60)
print("ВЫБОРОЧНОЕ СРЕДНЕЕ И СРЕДНЕКВАДРАТИЧЕСКОЕ ОТКЛОНЕНИЕ")
print("=" * 60)

for label, data in [("Исходная", df), ("Очищенная", df_clean)]:
    print(f"\n{label} выборка:")
    for col in ['ege_budg', 'total_income']:
        x = data[col].dropna().values
        mean = np.mean(x)
        std_pop = np.std(x, ddof=0)
        std_samp = np.std(x, ddof=1)
        var_pop = std_pop ** 2
        print(f"  {col}:")
        print(f"    Выборочное среднее (x̄): {mean:.4f}")
        print(f"    Выборочная дисперсия (D, ddof=0): {var_pop:.4f}")
        print(f"    Среднеквадратическое (σ, ddof=0): {std_pop:.4f}")
        print(f"    Среднеквадратическое (s, ddof=1): {std_samp:.4f}")


In [ ]:
# ============================================
# БЛОК 6: Ковариация и коэффициент корреляции Пирсона
# ============================================
print("\n" + "=" * 60)
print("КОВАРИАЦИЯ И КОЭФФИЦИЕНТ КОРРЕЛЯЦИИ")
print("=" * 60)

for label, data in [("Исходная", df), ("Очищенная", df_clean)]:
    print(f"\n{label} выборка:")
    mask = ~(np.isnan(data['ege_budg'].values) | np.isnan(data['total_income'].values))
    x = data['ege_budg'].values[mask]
    y = data['total_income'].values[mask]
    n = len(x)
    
    x_mean, y_mean = np.mean(x), np.mean(y)
    cov_formula = np.sum((x - x_mean) * (y - y_mean)) / n
    dx = np.sum((x - x_mean) ** 2) / n
    dy = np.sum((y - y_mean) ** 2) / n
    r_formula = cov_formula / np.sqrt(dx * dy) if dx * dy > 0 else 0
    r_scipy, p_value = stats.pearsonr(x, y)
    
    print(f"  n = {n}")
    print(f"  Ковариация (по формуле): {cov_formula:.4f}")
    print(f"  Ковариация (pandas): {data['ege_budg'].cov(data['total_income']):.4f}")
    print(f"  Коэффициент корреляции (scipy): {r_scipy:.4f}")
    print(f"  p-значение: {p_value:.6f}")
    print(f"  Статистически значима: {'Да' if p_value < 0.05 else 'Нет'}")


In [ ]:
# ============================================
# БЛОК 7: Интерпретация результатов
# ============================================
print("\n" + "=" * 60)
print("ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ")
print("=" * 60)

mask_clean = ~(np.isnan(df_clean['ege_budg'].values) | np.isnan(df_clean['total_income'].values))
x_clean = df_clean['ege_budg'].values[mask_clean]
y_clean = df_clean['total_income'].values[mask_clean]
r_clean, p_clean = stats.pearsonr(x_clean, y_clean)
cov_clean = np.cov(x_clean, y_clean, ddof=0)[0, 1]

if abs(r_clean) <= 0.3: strength = "слабая"
elif abs(r_clean) <= 0.7: strength = "умеренная"
else: strength = "сильная"
direction = "положительная" if r_clean > 0 else "отрицательная"

print(f"\nКоэффициент корреляции: {r_clean:.4f}, p-значение: {p_clean:.6f}")
print(f"Ковариация: {cov_clean:.2f}")
print(f"\nВЫВОД: Связь {strength} {direction}")
print(f"При {'росте' if r_clean > 0 else 'снижении'} среднего балла ЕГЭ доход ВУЗа {'растет' if r_clean > 0 else 'снижается'}.")
print(f"Связь {'статистически значима (p < 0.05)' if p_clean < 0.05 else 'НЕ статистически значима (p >= 0.05)'}.")


In [ ]:
# ============================================
# БЛОК 8: Доверительные интервалы (95% и 98%)
# ============================================
print("\n" + "=" * 60)
print("ДОВЕРИТЕЛЬНЫЕ ИНТЕРВАЛЫ")
print("=" * 60)

def conf_interval_mean(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data, ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf((1 + confidence) / 2, n - 1)
    margin = t_crit * std_err
    return mean, mean - margin, mean + margin, margin

def conf_interval_var(data, confidence=0.95):
    n = len(data)
    var = np.var(data, ddof=1)
    alpha = 1 - confidence
    chi2_low = stats.chi2.ppf(alpha / 2, n - 1)
    chi2_high = stats.chi2.ppf(1 - alpha / 2, n - 1)
    return var, (n - 1) * var / chi2_high, (n - 1) * var / chi2_low

print(f"\nОчищенная выборка (n={len(df_clean)}):")
for col, name in [('ege_budg', 'среднего балла ЕГЭ'), ('total_income', 'дохода ВУЗа')]:
    x = df_clean[col].dropna().values
    print(f"\n  Для {name}:")
    mean, ci95_l, ci95_h, m95 = conf_interval_mean(x, 0.95)
    _, ci98_l, ci98_h, m98 = conf_interval_mean(x, 0.98)
    print(f"    Матожидание 95%: [{ci95_l:.4f}, {ci95_h:.4f}] (±{m95:.4f})")
    print(f"    Матожидание 98%: [{ci98_l:.4f}, {ci98_h:.4f}] (±{m98:.4f})")
    _, var95_l, var95_h = conf_interval_var(x, 0.95)
    _, var98_l, var98_h = conf_interval_var(x, 0.98)
    print(f"    Дисперсия 95%: [{var95_l:.4f}, {var95_h:.4f}]")
    print(f"    Дисперсия 98%: [{var98_l:.4f}, {var98_h:.4f}]")

print("\n" + "=" * 60)
print("ЗАДАНИЕ ВЫПОЛНЕНО")
print("=" * 60)
